### Installing libraries

In [1]:
!pip install -q chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 104.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

### Load the model and libraries

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

model =  SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded


### Creating documents

In [3]:
documents = [
    "Python is a programming language used for AI development.",
    "Machine learning allows computers to learn patterns from data.",
    "ChromaDB stores embeddings and supports vector similarity search.",
    "Neural networks are an important part of deep learning.",
    "FastAPI is used to build APIs with Python.",
    "RAG combines document retrieval with language model generation.",
    "Next.js is a framework for building web applications.",
    "Embeddings represent text as numerical vectors."
]

ids = [f"doc_{i}" for i in range(len(documents))]

metadatas = [
    {"topic": "python"},
    {"topic": "machine_learning"},
    {"topic": "vector_database"},
    {"topic": "deep_learning"},
    {"topic": "backend"},
    {"topic": "rag"},
    {"topic": "frontend"},
    {"topic": "embeddings"}
]

print("Documents:", len(documents))


Documents: 8


### Converting documents into embeddings

In [4]:
embeddings = model.encode(documents,convert_to_numpy=True)

print("Number of embeddings", len(embeddings))
print("Vector dimensions:", embeddings.shape)
print("First vector:", embeddings[0][:10])

Number of embeddings 8
Vector dimensions: (8, 384)
First vector: [-0.05068963  0.0237672  -0.0136243   0.04116959 -0.01064518 -0.10861108
  0.0185456   0.05489112 -0.01132583  0.00202606]


### Connecting drive to save vector db

In [6]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


### Creating db

In [7]:
DB_PATH = "/content/drive/MyDrive/chroma_ai_db"

client = chromadb.PersistentClient(
    path=DB_PATH
)

print("Database path:", DB_PATH)

Database path: /content/drive/MyDrive/chroma_ai_db


### Creating collection

In [8]:
collection = client.get_or_create_collection(
    name="ai_learning",
    metadata={"hnsw:space": "cosine"}
)

print("Collection ready:", collection.name)

Collection ready: ai_learning


### Inserting Records to db

In [9]:
collection.upsert(
    ids=ids,
    documents=documents,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)

print("Records in database:", collection.count())

Records in database: 8


### Searching vector database

In [12]:
query = "What is RAG?"

# Convert the question into an embedding
query_embedding = model.encode(query).tolist()

# Search ChromaDB
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# Display results
for i, document in enumerate(results["documents"][0]):
    print(f"Result {i + 1}")
    print("Document:", document)
    print("Topic:", results["metadatas"][0][i]["topic"])
    print("Distance:", round(results["distances"][0][i], 4))
    print("-" * 50)

Result 1
Document: RAG combines document retrieval with language model generation.
Topic: rag
Distance: 0.5165
--------------------------------------------------
Result 2
Document: FastAPI is used to build APIs with Python.
Topic: backend
Distance: 0.7982
--------------------------------------------------
Result 3
Document: Next.js is a framework for building web applications.
Topic: frontend
Distance: 0.9149
--------------------------------------------------


### Retrieve records without searching the vector db

In [13]:
saved_records = collection.get(
    limit=5,
    include=["documents", "metadatas"]
)

for i, doc_id in enumerate(saved_records["ids"]):
    print("ID:", doc_id)
    print("Document:", saved_records["documents"][i])
    print("Metadata:", saved_records["metadatas"][i])
    print("-" * 40)

ID: doc_0
Document: Python is a programming language used for AI development.
Metadata: {'topic': 'python'}
----------------------------------------
ID: doc_1
Document: Machine learning allows computers to learn patterns from data.
Metadata: {'topic': 'machine_learning'}
----------------------------------------
ID: doc_2
Document: ChromaDB stores embeddings and supports vector similarity search.
Metadata: {'topic': 'vector_database'}
----------------------------------------
ID: doc_3
Document: Neural networks are an important part of deep learning.
Metadata: {'topic': 'deep_learning'}
----------------------------------------
ID: doc_4
Document: FastAPI is used to build APIs with Python.
Metadata: {'topic': 'backend'}
----------------------------------------


### Filter by metadata

In [20]:
python_docs = collection.get(
    where={"topic": "frontend"},
    include=["documents", "metadatas"]
)

print(python_docs["documents"])

['Next.js is a framework for building web applications.']


### checking for Reopening the collection

In [21]:
new_client = chromadb.PersistentClient(
    path=DB_PATH
)

loaded_collection = new_client.get_collection(
    name="ai_learning"
)

print("Collection:", loaded_collection.name)
print("Saved records:", loaded_collection.count())

Collection: ai_learning
Saved records: 8
